# 07 · CD con los parámetros del compensador (CDC)

**Objetivo.** Probar el emulador de dispersión cromática con el punto de diseño que usa el grupo del compensador (CDC por overlap-save en frecuencia), para:

- verificar que el bloque `ChromaticDispersion` reproduce la CD ideal con esos parámetros (60 GBd, 72 GSa/s, 200 km);
- comprobar que un compensador con su ventana (FFT de 384 muestras, avance de 192) recupera la señal, con y sin ruido;
- medir el **residuo de β₃** cuando el CDC compensa solo β₂ (el grupo presentó resultados con β₃);
- dejar explícita la **convención de signo** que tiene que usar el CDC ($H^*$, decisión 2) y el **alcance** de su ventana;
- contrastar el emulador con **OptiCommPy** (canal `linearFiberChannel` y compensador `edc`), la librería abierta de referencia del área. Los gráficos de constelación, ojo y espectro usan también sus funciones (`constHist`, `eyediagram`, `plotPSD`).

**Requiere OptiCommPy** (`pip install OptiCommPy`); todavía no está en las dependencias del paquete.

Parámetros tomados de la presentación del grupo del CDC ("Parámetros del sistema simulado").

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import resample_poly

import mpl_scatter_density  # noqa: F401  (proyección "scatter_density" que usa constHist)
import scipy.constants as const
from optic.dsp.equalization import edc
from optic.models.channels import linearFiberChannel
from optic.plot import constHist, eyediagram, plotPSD
from optic.utils import parameters

from emulador import plots
from emulador.blocks import ChromaticDispersion
from emulador.cd import cd_N, cd_R, cd_hbar, cd_overlap_save, cd_reference, cd_summary
from emulador.config import D_si, S_si
from scipy.optimize import minimize_scalar
from emulador.metrics import evm_opt, nrmse
from emulador.stimuli import map_gray, prbs

plots.use_style()

## 1. Parámetros del grupo del CDC

| Bloque | Parámetro | Valor |
|---|---|---|
| Señal | Tasa de símbolos | 60 GBd |
| | Frecuencia de muestreo | 72 GSa/s (sobremuestreo 1,2) |
| | Modulación | 16-QAM Gray |
| | Largo | 10 000 símbolos |
| | Conformador | coseno realzado, roll-off 0,1, 300 símbolos |
| Canal | Fibra | 200 km, D = 17 ps/(nm·km), λ = 1550 nm |
| | Pendiente de dispersión | S = 0,057 ps/(nm²·km) (β₃; no figura en la captura, pero el grupo presentó resultados con β₃) |
| | Ruido | OSNR 20 dB |
| CDC | Ventana FFT (B) | 384 muestras |
| | Avance por ciclo (H) | 192 muestras (50 % de solapamiento) |
| | Taps del ecualizador (F) | 256 (fuera del alcance de esta prueba) |

**Equivalencia con el emulador.** El overlap-save del emulador usa una FFT de $L = 2N$ con $N$ muestras nuevas por bloque: la ventana del CDC (B = 384, H = 192) es exactamente $L = 384$, $N = 192$. El emulador acepta ese $N$ explícito (tiene que ser par y $N \ge R$).

**Diferencias con el escenario del emulador.**
- 1,2 muestras por símbolo: el `Scenario` del emulador solo admite 1, 2, 4, 8 o 16 sps, pero el bloque de CD recibe $f_s$ directamente y no depende de eso. La señal se genera acá (sección 3).
- Coseno realzado (RC) completo en el transmisor, no raíz (RRC): la señal ya cumple Nyquist y el receptor muestrea sin filtro adaptado.
- N = 192 no es potencia de 2; `cd_N` elegiría 256 para el hardware del emulador (decisión 5). Para el CDC es una elección de su diseño.
- β₃: la captura no lo incluye, pero el grupo presentó resultados con él. Se usa S = 0,057 ps/(nm²·km), el valor típico de SSMF (parámetro `S_PS`, ajustable si usaron otro).

## 2. Parámetros y magnitudes derivadas

In [ ]:
RS = 60e9
FS = 72e9
OS = FS / RS                       # 1,2
ROLLOFF, SPAN = 0.1, 300           # RC del transmisor
N_SYM = 10_000
D_PS, LAM_NM, Z_KM = 17.0, 1550.0, 200.0
S_PS = 0.057                       # pendiente de dispersión [ps/(nm²·km)] -> β₃
OSNR_DB = 20.0
N_CDC = 192                        # avance H; ventana B = 2·N = 384

D, lam, z = D_si(D_PS), LAM_NM / 1e9, Z_KM * 1e3
S = S_si(S_PS)
R = cd_R(FS, D, lam, z)


class _Esc:  # cd_summary solo usa fs, Rs y wavelength
    fs, Rs, wavelength = FS, RS, lam


s = cd_summary(_Esc, D, z, S)
print(f"sobremuestreo = {OS:.1f} muestras/símbolo")
print(f"R = {R} muestras  (chirp de la CD a {FS / 1e9:.0f} GSa/s y {Z_KM:.0f} km)")
print(f"N del CDC = {N_CDC} (ventana {2 * N_CDC}); N >= R: {N_CDC >= R}; margen = {N_CDC - R} muestras")
print(f"cd_N (potencia de 2 del emulador) = {cd_N(FS, D, lam, z)}")
print(f"retardo del overlap-save = N/2 = {N_CDC // 2} muestras = {N_CDC / 2 / FS * 1e12:.0f} ps")
print(f"β₂ = {s['beta2_ps2_km']:.2f} ps²/km, β₃ = {s['beta3_ps3_km']:.3f} ps³/km")
print(f"dispersión acumulada = {s['Dz_ps_nm']:.0f} ps/nm; pendiente del retardo de grupo = {s['gd_slope_ps_GHz']:.1f} ps/GHz")
print(f"ensanchamiento en el borde de la señal (±{(1 + ROLLOFF) * RS / 2e9:.0f} GHz): "
      f"±{abs(s['gd_slope_ps_GHz']) * (1 + ROLLOFF) * RS / 2e9:.0f} ps = ±{abs(s['gd_slope_ps_GHz']) * (1 + ROLLOFF) * RS / 2e9 * 1e-12 * RS:.1f} símbolos")

## 3. Señal: 16-QAM Gray con coseno realzado a 1,2 sps

Los símbolos salen de la PRBS-15 del emulador con mapeo Gray (potencia media 1; en la presentación del CDC los niveles no están normalizados, ±1 y ±3, lo que solo cambia la escala). Se conforman con un RC de roll-off 0,1 y 300 símbolos a 6 sps y se toma una de cada 5 muestras: 6/5 = 1,2 sps. No hay aliasing porque la señal ocupa $\pm(1+\rho)R_s/2$ = ±33 GHz y Nyquist está en ±36 GHz.

El receptor de esta notebook interpola por 5 (vuelve a 6 sps) y toma una muestra por símbolo; la EVM se mide con ganancia compleja óptima (`metrics.evm_opt`), descartando 200 símbolos en cada borde.

In [ ]:
UP, DOWN = 6, 5          # 6 sps -> /5 -> 1,2 sps
DISCARD = 200            # símbolos descartados por borde en la EVM


def rc_taps(beta, span, sps):
    # Coseno realzado (no raíz): h(t) = sinc(t)·cos(πβt) / (1 - (2βt)²), t en símbolos
    t = np.arange(-span * sps // 2, span * sps // 2 + 1) / sps
    h = np.sinc(t) * np.cos(np.pi * beta * t)
    den = 1 - (2 * beta * t) ** 2
    sing = np.isclose(den, 0)
    h[~sing] /= den[~sing]
    h[sing] = np.pi / 4 * np.sinc(1 / (2 * beta))
    return h


def rx_symbols(y):
    # 1,2 sps -> 6 sps (interpolación polifásica) y una muestra por símbolo
    return resample_poly(y, DOWN, 1)[: N_SYM * UP][::UP]


def evm_grid(y):
    # EVM muestreando en la grilla fija (sin corregir retardos fraccionarios)
    return evm_opt(rx_symbols(y), sym, DISCARD)


def evm_ideal_timing(y):
    # EVM con el mejor retardo fraccionario continuo (recuperación de temporización ideal).
    # β₃ agrega un retardo global de una fracción de muestra; un retardo puro no es
    # distorsión, así que se busca en [-0,5, 0,5] muestras y se refina (decisión 23).
    Y = np.fft.fft(y)
    f = np.fft.fftfreq(y.size)
    at = lambda t: evm_opt(rx_symbols(np.fft.ifft(Y * np.exp(2j * np.pi * f * t))), sym, DISCARD)  # noqa: E731
    taus = np.linspace(-0.5, 0.5, 41)
    e = [at(t) for t in taus]
    i = int(np.argmin(e))
    fine = minimize_scalar(at, bounds=(taus[i] - 0.025, taus[i] + 0.025), method="bounded", options={"xatol": 1e-4})
    return min(fine.fun, e[i])


sym = map_gray(prbs(15, 4 * N_SYM), "qam16")
up = np.zeros(N_SYM * UP, complex)
up[::UP] = sym
x6 = np.convolve(up, rc_taps(ROLLOFF, SPAN, UP), mode="same")
x = x6[::DOWN]
x = x / np.sqrt(np.mean(np.abs(x) ** 2))
evm_b2b = evm_opt(rx_symbols(x), sym, DISCARD)

fig = plt.figure(figsize=(10, 3.8))
plotPSD(x, Fs=FS, NFFT=2048, fig=fig, label="16-QAM, RC 0,1")      # OptiCommPy
ax = fig.gca()
for sgn in (-1, 1):
    ax.axvline(sgn * (1 + ROLLOFF) * RS / 2, ls="--", color="C1", lw=1)
    ax.axvline(sgn * FS / 2, ls=":", color="k", lw=1)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v / 1e9:.0f}"))
ax.set_xlabel("f [GHz]")
ax.set_title("16-QAM, 60 GBd, RC roll-off 0,1, 72 GSa/s (-- : ±33 GHz; ··· : ±fs/2)")
plots.save_fig(fig, "07_espectro")
print(f"{x.size} muestras; EVM back-to-back = {100 * evm_b2b:.2f} %")

**Qué muestra la figura.** La señal ocupa ±33 GHz (líneas de trazos) y no llega a ±36 GHz (Nyquist a 72 GSa/s, punteadas): el sobremuestreo de 1,2 alcanza sin aliasing, con solo 3 GHz de margen a cada lado. Sin canal, la EVM es el piso de esta cadena (RC truncado e interpolación del receptor).

## 4. CD de 200 km con el emulador

Se aplica el bloque `ChromaticDispersion` con β₂ y β₃, con N = 192 (la ventana del CDC) y con N = 256 (`cd_N`), y se compara con la referencia ideal (`cd_reference`, una FFT de toda la señal). Se descartan N muestras por borde.

In [ ]:
y_cd, err_cd = {}, {}
ref = cd_reference(x, FS, D, lam, z, S=S)
for N in (N_CDC, cd_N(FS, D, lam, z)):
    blk = ChromaticDispersion(fs=FS, D_ps_nm_km=D_PS, wavelength_nm=LAM_NM, length_km=Z_KM, S_ps_nm2_km=S_PS, N=N)
    y_cd[N] = blk(x)
    err_cd[N] = nrmse(y_cd[N][N:-N], ref[N:-N])
    print(f"N = {N}: NRMSE contra la referencia = {err_cd[N]:.1e}")
evm_cd = evm_opt(rx_symbols(y_cd[N_CDC]), sym, DISCARD)
print(f"EVM después de la CD (sin compensar) = {100 * evm_cd:.1f} %")

## 5. Compensación con la ventana del CDC

El compensador se emula con el mismo overlap-save (B = 384, H = 192) y el filtro conjugado, que es la CD con $-D$ (y $-S$). Es un CDC ideal en punto flotante: sirve de referencia para el diseño del grupo. Se prueban dos compensadores:

- **CDC solo β₂** ($-D$, $S = 0$): lo habitual en un CDC; deja la ISI residual de β₃ para el ecualizador.
- **CDC β₂ + β₃** ($-D$, $-S$): el conjugado completo; muestra el error propio de la ventana.

Y además: con ruido de OSNR 20 dB entre la fibra y el CDC, y con el **signo equivocado** ($H$ en lugar de $H^*$).

**Temporización.** β₃ agrega un retardo global de una fracción de muestra. Se reporta la EVM con la grilla fija y con temporización ideal (el mejor retardo fraccionario), como en la decisión 23: la diferencia es error de temporización del receptor, no ISI.

**Ruido.** El bloque AWGN del emulador todavía no está en esta rama (lo implementa otra tesis), así que se agrega acá. La OSNR se mide en 12,5 GHz; para una polarización, $E_s/N_0 = \mathrm{OSNR}\cdot 2B_{ref}/R_s$: OSNR 20 dB equivale a $E_s/N_0$ ≈ 16,2 dB a 60 GBd. (Si en la simulación del CDC se usa el valor de 20 dB directamente como SNR, ese caso también se muestra.)

In [ ]:
B_REF = 12.5e9


def add_noise(y, esn0_db, seed=1):
    # ruido complejo blanco en toda la banda fs; Es/N0 referido a la tasa de símbolos
    rng = np.random.default_rng(seed)
    ps = np.mean(np.abs(y) ** 2)
    sigma2 = ps * (FS / RS) / 10 ** (esn0_db / 10)
    return y + np.sqrt(sigma2 / 2) * (rng.standard_normal(y.size) + 1j * rng.standard_normal(y.size))


def cdc_block(S_ps, sign=-1):
    # sign = -1: conjugado (H*), el correcto; sign = +1: signo equivocado (H)
    return ChromaticDispersion(fs=FS, D_ps_nm_km=sign * D_PS, wavelength_nm=LAM_NM, length_km=Z_KM,
                               S_ps_nm2_km=sign * S_ps, N=N_CDC)


esn0_osnr = OSNR_DB + 10 * np.log10(2 * B_REF / RS)
cdc_b2, cdc_full = cdc_block(0.0), cdc_block(S_PS)
y = y_cd[N_CDC]
cases = {
    "Sin canal": x,
    "CD 200 km": y,
    "CDC solo β₂": cdc_b2(y),
    "CDC β₂ + β₃": cdc_full(y),
    "OSNR 20 dB + CDC solo β₂": cdc_b2(add_noise(y, esn0_osnr)),
    "OSNR 20 dB + CDC β₂ + β₃": cdc_full(add_noise(y, esn0_osnr)),
    "SNR 20 dB + CDC β₂ + β₃": cdc_full(add_noise(y, OSNR_DB)),
    "CDC con signo equivocado (H)": cdc_block(S_PS, sign=+1)(y),
}
evm = {k: evm_grid(v) for k, v in cases.items()}
evm_it = {k: evm_ideal_timing(cases[k]) for k in ("CDC solo β₂", "CDC β₂ + β₃")}

show = ["Sin canal", "CD 200 km", "CDC solo β₂", "CDC β₂ + β₃", "OSNR 20 dB + CDC β₂ + β₃"]
# constelaciones como mapa de densidad (OptiCommPy, constHist). Sin constrained_layout:
# calcula los márgenes antes de que el mapa de densidad tenga extensión.
fig = plt.figure(figsize=(22, 4.6), layout="none")
for i, k in enumerate(show):
    ax = fig.add_subplot(1, len(show), i + 1, projection="scatter_density")
    s_rx = rx_symbols(cases[k])[DISCARD:-DISCARD]
    ref_s = sym[DISCARD:-DISCARD]
    g = np.vdot(s_rx, ref_s) / np.vdot(s_rx, s_rx)        # ganancia compleja óptima, como evm_opt
    constHist(g * s_rx, ax)
    lim = 1.3 if k != "CD 200 km" else 1.15 * np.max(np.abs(np.r_[(g * s_rx).real, (g * s_rx).imag]))
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect("equal")
    ax.set_title(f"{k} · EVM {100 * evm[k]:.2f} %")
fig.subplots_adjust(left=0.03, right=0.99, wspace=0.25)
fig.canvas.draw()  # el mapa de densidad necesita dibujarse antes del recorte "tight"
plots.save_fig(fig, "07_constelaciones")

# diagramas de ojo de la componente en fase (OptiCommPy, eyediagram). eyediagram cubre
# unos (muestras / 1024) símbolos: se interpola a 48 sps para ver ~470 símbolos.
SPS_EYE = 48
fig, axs = plt.subplots(1, 3, figsize=(16, 4.3), sharey=True)
for ax, k in zip(axs, ["Sin canal", "CDC β₂ + β₃", "OSNR 20 dB + CDC β₂ + β₃"]):
    y48 = resample_poly(cases[k], SPS_EYE * DOWN // UP, 1)        # 1,2 sps -> 48 sps
    y48 = y48[DISCARD * SPS_EYE + SPS_EYE // 2 :]                 # sin bordes; ojo centrado
    y48 = y48 / np.sqrt(np.mean(np.abs(y48) ** 2))
    eyediagram(y48.real, y48.size, SPS_EYE, n=2, ptype="fancy", ax=ax, show=False)
    ax.set_title(k)
plots.save_fig(fig, "07_ojos", dpi=110)

print(f"Es/N0 equivalente a OSNR {OSNR_DB:.0f} dB: {esn0_osnr:.1f} dB")
for k, e in evm.items():
    extra = f"   temporización ideal: {100 * evm_it[k]:.2f} %" if k in evm_it else ""
    print(f"{k:32s} EVM = {100 * e:6.2f} %  ({-20 * np.log10(e):5.1f} dB){extra}")

**Qué muestran las figuras.** Constelaciones como mapa de densidad (`constHist` de OptiCommPy): sin canal, la constelación es limpia. Después de 200 km de CD es una nube sin estructura (el retardo de grupo llega a ±0,9 ns en el borde de la banda: cada símbolo recibido mezcla unos ±54 símbolos vecinos). El CDC con la ventana de 384 la recupera. Si compensa solo β₂, queda el residuo de β₃ (y un pequeño retardo fraccionario); con el conjugado completo, solo el error de la ventana. Con ruido de OSNR 20 dB, la EVM la domina el ruido y la diferencia entre los dos CDC desaparece. Los ojos (`eyediagram` de OptiCommPy, componente en fase) muestran lo mismo: después del CDC el ojo queda tan abierto como sin canal, y el ruido lo cierra parcialmente.

## 6. Validación cruzada con OptiCommPy

OptiCommPy modela la CD con `linearFiberChannel` y la compensa con `edc`. Se compara con el emulador sin pérdidas ($\alpha = 0$) y solo con β₂ (OptiCommPy no incluye β₃), a 200 km y 72 GSa/s.

**Convención de signo.** OptiCommPy aplica $\exp\left(+j\frac{\beta_2}{2}\omega^2 L\right)$ con $\beta_2 = -D\lambda^2/(2\pi c)$, o sea $H(f) = \exp(-j\pi D\lambda^2 L f^2/c)$: **el signo opuesto** al del emulador (decisión 2). Las dos convenciones son válidas por separado (dependen del signo de la transformada de Fourier), pero no se pueden mezclar: el `edc` de OptiCommPy invierte el canal de OptiCommPy, no el del emulador.

In [ ]:
def ocp_params(D_ps):
    p = parameters()
    p.L, p.D, p.alpha = Z_KM, D_ps, 0.0
    p.Fc, p.Fs, p.Rs = const.c / lam, FS, RS
    return p


def evm_search(y):
    # EVM buscando la fase de muestreo (6 sps) y el retardo entero: edc agrega un retardo propio
    y6 = resample_poly(y, DOWN, 1)
    best = np.inf
    for ph in range(UP):
        s_ = y6[ph::UP]
        lag = int(np.argmax(np.abs(np.correlate(s_[:N_SYM], sym, mode="full")))) - (N_SYM - 1)
        a = s_[max(lag, 0) : max(lag, 0) + N_SYM]
        r_ = sym[max(-lag, 0) : max(-lag, 0) + a.size]
        best = min(best, evm_opt(a[: r_.size], r_, DISCARD))
    return best


y_ocp = linearFiberChannel(x.reshape(-1, 1), ocp_params(D_PS)).ravel()
ref_pos = cd_reference(x, FS, D, lam, z)          # emulador, convención positiva
ref_neg = cd_reference(x, FS, -D, lam, z)         # emulador con el signo de OptiCommPy
m = 1000                                          # OptiCommPy usa una FFT circular: sin bordes
xval = {"OptiCommPy contra emulador (+D)": nrmse(y_ocp[m:-m], ref_pos[m:-m]),
        "OptiCommPy contra emulador (−D)": nrmse(y_ocp[m:-m], ref_neg[m:-m])}
y_emu = ChromaticDispersion(fs=FS, D_ps_nm_km=D_PS, wavelength_nm=LAM_NM, length_km=Z_KM, N=N_CDC)(x)
edc_cases = {
    "canal OptiCommPy + edc(D = +17)": edc(y_ocp.reshape(-1, 1), ocp_params(+D_PS)).ravel(),
    "canal emulador + edc(D = +17)": edc(y_emu.reshape(-1, 1), ocp_params(+D_PS)).ravel(),
    "canal emulador + edc(D = −17)": edc(y_emu.reshape(-1, 1), ocp_params(-D_PS)).ravel(),
}
evm_edc = {k: evm_search(v) for k, v in edc_cases.items()}
for k, v in xval.items():
    print(f"{k:34s} NRMSE = {v:.1e}")
print()
for k, v in evm_edc.items():
    print(f"{k:34s} EVM = {100 * v:6.2f} %")

**Lectura.** El canal de OptiCommPy coincide con el emulador aplicado con **−D**, no con +D: misma física, signo opuesto. Por eso el `edc` de OptiCommPy con D = +17 compensa su propio canal, pero sobre la salida del emulador **duplica** la dispersión. **Para compensar los vectores del emulador con OptiCommPy hay que llamar a `edc` con D negativo** (o, equivalentemente, conjugar la señal antes y después). Es el aviso de la decisión 2 para la tesis de CDC.

## 7. Alcance de la ventana del CDC y residuo de β₃ con la distancia

¿Hasta qué distancia alcanza la ventana de 384 (N = 192)? Se barre z con el canal ideal (β₂ y β₃) y se mide el NRMSE del emulador con N = 192 contra la referencia, y la EVM después de compensar con la ventana de 384: con el conjugado completo (error de la ventana) y solo β₂ con temporización ideal (residuo de β₃). Sin ruido. La condición de diseño es $N \ge R$.

In [ ]:
Z_SWEEP = np.arange(25, 401, 25)
nrmse_z, evm_full_z, evm_b2_z, R_z = [], [], [], []
for zk in Z_SWEEP:
    zz = zk * 1e3
    R_z.append(cd_R(FS, D, lam, zz))
    ch = cd_reference(x, FS, D, lam, zz, S=S)       # canal ideal (verdad física)
    m = N_CDC
    # emulador con N = 192 (funciones, porque el bloque exige N >= R) contra la referencia
    ols = cd_overlap_save(x, cd_hbar(N_CDC, FS, D, lam, zz, S=S))[N_CDC // 2 : N_CDC // 2 + x.size]
    nrmse_z.append(nrmse(ols[m:-m], ch[m:-m]))
    # CDC con la ventana de 384 sobre el canal ideal, para aislar el error del CDC
    for S_c, out in ((-S, evm_full_z), (0.0, evm_b2_z)):
        comp = cd_overlap_save(ch, cd_hbar(N_CDC, FS, -D, lam, zz, S=S_c))[N_CDC // 2 : N_CDC // 2 + x.size]
        out.append(evm_grid(comp) if S_c != 0.0 else evm_ideal_timing(comp))
z_max = max(zk for zk in range(1, 1000) if cd_R(FS, D, lam, zk * 1e3) <= N_CDC)  # último km con R <= N

fig, axs = plt.subplots(1, 2, figsize=(13, 4.3))
axs[0].semilogy(Z_SWEEP, nrmse_z, "o-")
axs[0].axhline(1e-2, ls=":", color="k")
axs[0].set_title("Emulador (N = 192) contra la referencia: NRMSE")
axs[1].semilogy(Z_SWEEP, 100 * np.array(evm_full_z), "o-", color="C0", label="CDC β₂ + β₃")
axs[1].semilogy(Z_SWEEP, 100 * np.array(evm_b2_z), "s--", color="C1", label="CDC solo β₂ (temporiz. ideal)")
axs[1].axhline(100 * evm_b2b, ls=":", color="k", label="piso sin canal")
axs[1].set_title("CDC (ventana 384) sobre la CD ideal: EVM [%]")
axs[1].legend()
for ax in axs:
    ax.axvline(z_max, ls="--", color="C2")
    ax.set_xlabel("z [km]")
plots.save_fig(fig, "07_alcance_ventana")

print(f"N = {N_CDC} cubre R hasta z = {z_max} km (R({z_max} km) = {cd_R(FS, D, lam, z_max * 1e3)}, "
      f"R({z_max + 1} km) = {cd_R(FS, D, lam, (z_max + 1) * 1e3)})")
print(" z [km]    R   NRMSE emul.   EVM CDC β₂+β₃   EVM CDC solo β₂")
for zk, r_, e1, e2, e3 in zip(Z_SWEEP, R_z, nrmse_z, evm_full_z, evm_b2_z):
    print(f"{zk:6d} {r_:5d}   {e1:10.1e}   {100 * e2:12.2f} %   {100 * e3:13.2f} %")

**Qué muestra la figura.** Mientras $R \le 192$, es decir hasta 271 km (línea de trazos), el emulador y el CDC completo tienen error chico; cuando el chirp supera la ventana, el error sube de golpe, igual que en la Fig. 9 del notebook 04. La curva del CDC solo β₂ queda por encima: es el residuo de β₃, que crece con la distancia aunque la ventana alcance. Los 200 km del escenario quedan dentro del alcance, con margen.

## 8. Validación

In [ ]:
assert R == 141 and N_CDC >= R
assert evm_b2b < 0.005
assert err_cd[N_CDC] < 1e-2 and err_cd[256] < err_cd[N_CDC]
assert evm_cd > 0.5
assert evm["CDC β₂ + β₃"] < 0.01
assert evm_it["CDC solo β₂"] > evm_it["CDC β₂ + β₃"]                 # queda el residuo de β₃
assert evm["CDC solo β₂"] >= evm_it["CDC solo β₂"]                    # parte es temporización
assert evm["CDC con signo equivocado (H)"] > 0.5
assert evm["OSNR 20 dB + CDC β₂ + β₃"] > 10 * evm["CDC β₂ + β₃"]
assert z_max == 271
assert all(b >= a for a, b in zip(evm_full_z, evm_b2_z))
assert xval["OptiCommPy contra emulador (−D)"] < 1e-3 < 1 < xval["OptiCommPy contra emulador (+D)"]
assert evm_edc["canal OptiCommPy + edc(D = +17)"] < 0.01 and evm_edc["canal emulador + edc(D = −17)"] < 0.01
assert evm_edc["canal emulador + edc(D = +17)"] > 0.5
print("OK")

## 9. Conclusiones

- **El emulador funciona con el punto de diseño del CDC.** A 72 GSa/s y 200 km el chirp ocupa R = 141 muestras; la ventana del CDC (B = 384, H = 192) equivale al overlap-save del emulador con N = 192 ≥ R. Con ese N y con β₃ (S = 0,057 ps/(nm²·km)), el bloque `ChromaticDispersion` sigue a la CD ideal con NRMSE 3,2e-3 (1,8e-3 con N = 256, el que elegiría `cd_N`).
- **Sin compensar, 200 km destruyen la señal**: EVM 99,5 %. El retardo de grupo llega a ±0,9 ns en el borde de la banda, unos ±54 símbolos.
- **CDC con el conjugado completo (β₂ + β₃)**: EVM 0,33 % (piso de esta cadena sin canal: 0,15 %). Es el error propio de la ventana de 384.
- **CDC solo β₂**: EVM 0,95 % en la grilla fija y 0,48 % con temporización ideal. La diferencia con el CDC completo (0,48 % contra 0,33 %, SNR equivalente ~46 dB) es la ISI residual de β₃; el resto (0,95 % contra 0,48 %) es el pequeño retardo fraccionario que agrega β₃ y que corrige la recuperación de temporización. A 60 GBd y 200 km el residuo de β₃ es chico; crece con la distancia (0,25 % a 100 km, 0,69 % a 250 km) y, sobre todo, con la tasa (notebook 04).
- **Con OSNR 20 dB** (Es/N0 ≈ 16,2 dB para una polarización a 60 GBd) la EVM después del CDC es 16,2 % con los dos compensadores: domina el ruido, y el residuo de β₃ no se ve. Si el valor de 20 dB se usa como SNR directamente, la EVM es 10,5 %: conviene acordar con el grupo cuál de las dos definiciones usan.
- **Signo**: compensar con $H$ en lugar de $H^*$ duplica la dispersión (EVM 99,7 %). Con la convención positiva del emulador (decisión 2), el CDC tiene que usar el conjugado, también para β₃ ($-S$).
- **OptiCommPy usa la convención de signo opuesta**: su canal coincide con el emulador aplicado con −D, y su `edc` con D = +17 duplica la dispersión de los vectores del emulador (EVM ~100 %). Para usar OptiCommPy como CDC sobre el emulador, `edc` va con **D = −17**. Si el grupo del CDC trabaja con OptiCommPy, es el punto a coordinar.
- **Alcance de la ventana**: N = 192 cubre R hasta **271 km** a 72 GSa/s. Más allá, el error sube de golpe (EVM del CDC 0,8 % a 275 km, 1,9 % a 300 km, 13 % a 400 km) y domina sobre el residuo de β₃. Para distancias mayores, la ventana tiene que crecer con R ∝ z·f_s².
- **Para el grupo del CDC**: el emulador puede generar sus vectores de prueba con estos parámetros (señal a 1,2 sps, bloque de CD con `fs = 72e9` y `S_ps_nm2_km = 0.057`); la salida cruda `process_hw` sale retrasada N/2 = 96 muestras si se usa N = 192.

Figuras en `docs/golden_model/figuras/07_*.png` (constelaciones, ojos y espectro con funciones de OptiCommPy).